# California Housing

This is the dataset used in the second chapter of Aurélien Géron's recent book 'Hands-On Machine learning with Scikit-Learn and TensorFlow'. It serves as an excellent introduction to implementing machine learning algorithms because it requires rudimentary data cleaning, has an easily understandable list of variables and sits at an optimal size between being to toyish and too cumbersome.

The data contains information from the 1990 California census. So although it may not help you with predicting current housing prices like the Zillow Zestimate dataset, it does provide an accessible introductory dataset for teaching people about the basics of machine learning.

The data pertains to the houses found in a given California district and some summary stats about them based on the 1990 census data. Be warned the data aren't cleaned so there are some preprocessing steps required! The columns are as follows, their names are pretty self explanitory:

<hr />

## EDA


In [11]:
import pandas as pd

data = pd.read_csv('./data/CH/housing.csv')
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.6 MB


In [12]:
data.head()


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [13]:
Faltantes = {
    'Numero de valores faltantes: ': data.isna().sum(),
    'Porcentaje de faltantes: ': (data.isna().mean() * 100).round(2)
}

print('='*50)
print('Numero de registros: ', len(data))
print(Faltantes)
print('='*50)

data.dropna(inplace=True)


Numero de registros:  20640
{'Numero de valores faltantes: ': longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64, 'Porcentaje de faltantes: ': longitude             0.0
latitude              0.0
housing_median_age    0.0
total_rooms           0.0
total_bedrooms        1.0
population            0.0
households            0.0
median_income         0.0
median_house_value    0.0
ocean_proximity       0.0
dtype: float64}


### Resumen
Despues de observar la cantidad de valores faltantes, se tomo la desicion de solamente eliminar los datos los cuales carecen de la variable **total_bedrooms** puesto que solo representan un 1% de la cantidad de datos totales que se tienen y el retirarlos no causara menor problema.


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_percentage_error

x = data.drop(columns=['median_house_value'])
y = data['median_house_value']

x_train, x_test, y_train, y_test = train_test_split(x,y, train_size=0.8, random_state=777)

## Preparacion de caracteristicas para todos los modelos puesto que a todos se les van a aplicar las mismas transformaciones
num_cols_no_normales = ['longitude','latitude', ]
num_cols_si = ['housing_median_age','total_rooms','total_bedrooms','population','households','median_income']
cat_cols = ['ocean_proximity']

preprossesor = ColumnTransformer(
    transformers=[
        ('num_cols_no_normales', MinMaxScaler(), num_cols_no_normales),
        ('num_cols_si', StandardScaler(), num_cols_si),
        ('cat_cols', OneHotEncoder(drop='first'), cat_cols)
    ],
    remainder='passthrough'
)

## Linear Regression

In [15]:
from sklearn.linear_model import LinearRegression

## Creacion del modelo con pipeline
lr = Pipeline(steps=[
    ('preprossesor', preprossesor),
    ('modelo', LinearRegression(n_jobs=-1))
])


## Entrenamiento del modelo y prediccion
lr.fit(x_train, y_train)
y_pred_lr = lr.predict(x_test)

## Metricas
mae_lr = mean_absolute_error(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = root_mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)
mape_lr = mean_absolute_percentage_error(y_test, y_pred_lr)

# 2. Calculo del R2 Ajustado
# n = numero de muestras en el set de prueba
# p = numero de características (predictores) independientes
n = x_test.shape[0]
p = x_test.shape[1] 

r2_ajustada_lr = 1 - ((1 - r2_lr) * (n - 1) / (n - p - 1))


print(f"MAE: {mae_lr:.4f}")
print(f"MSE: {mse_lr:.4f}")
print(f"RMSE: {rmse_lr:.4f}")
print(f"MAPE: {mape_lr:.4f} (o {mape_lr*100:.2f}%)")
print(f"R²: {r2_lr:.4f}")
print(f"R² Ajustadof: {r2_ajustada_lr:.4f}")

MAE: 49193.4510
MSE: 4582634774.4443
RMSE: 67695.1606
MAPE: 0.2833 (o 28.33%)
R²: 0.6473
R² Ajustadof: 0.6465


##  K Nearest Neighbors Regressor

In [16]:
from sklearn.neighbors import KNeighborsRegressor

knr = Pipeline(steps=[
    ('preprocessor', preprossesor),
    ('modelo', KNeighborsRegressor(n_neighbors= 5, n_jobs= -1))
])

## Entrenamiento del modelo yy prediccion
knr.fit(x_train, y_train)
y_pred_knr = knr.predict(x_test)

## Metricas

mae_knr = mean_absolute_error(y_test, y_pred_knr)
mse_knr = mean_squared_error(y_test, y_pred_knr)
rmse_knr = root_mean_squared_error(y_test, y_pred_knr)
r2_knr = r2_score(y_test, y_pred_knr)
mape_knr = mean_absolute_percentage_error(y_test, y_pred_knr)

# 2. Calculo del R2 Ajustado
# n = numero de muestras en el set de prueba
# p = numero de características (predictores) independientes
n = x_test.shape[0]
p = x_test.shape[1] 

r2_ajustada_knr = 1 - ((1 - r2_knr) * (n - 1) / (n - p - 1))


print(f"MAE: {mae_knr:.4f}")
print(f"MSE: {mse_knr:.4f}")
print(f"RMSE: {rmse_knr:.4f}")
print(f"MAPE: {mape_knr:.4f} (o {mape_knr*100:.2f}%)")
print(f"R²: {r2_knr:.4f}")
print(f"R² Ajustado: {r2_ajustada_knr:.4f}") 

MAE: 43599.7779
MSE: 4060119921.1013
RMSE: 63719.0703
MAPE: 0.2426 (o 24.26%)
R²: 0.6875
R² Ajustado: 0.6868


## Random Forest Regressor

In [17]:
from sklearn.ensemble import RandomForestRegressor

rfr = Pipeline(steps=[
    ('preprocessor', preprossesor),
    ('modelo', RandomForestRegressor(n_estimators= 500, random_state=777, n_jobs=-1))
])

## Entrenamiento del modelo y prediccion
rfr.fit(x_train, y_train)
y_pred_rfr = rfr.predict(x_test)

## Metricas
mae_rfr = mean_absolute_error(y_test, y_pred_rfr)
mse_rfr= mean_squared_error(y_test, y_pred_rfr)
rmse_rfr = root_mean_squared_error(y_test, y_pred_rfr)
r2_rfr = r2_score(y_test, y_pred_rfr)
mape_rfr = mean_absolute_percentage_error(y_test, y_pred_rfr)

# 2. Calculo del R2 Ajustado
# n = numero de muestras en el set de prueba
# p = numero de características (predictores) independientes
n = x_test.shape[0]
p = x_test.shape[1] 

r2_ajustada_rfr = 1 - ((1 - r2_rfr) * (n - 1) / (n - p - 1))


print(f"MAE: {mae_rfr:.4f}")
print(f"MSE: {mse_rfr:.4f}")
print(f"RMSE: {rmse_rfr:.4f}")
print(f"MAPE: {mape_rfr:.4f} (o {mape_rfr*100:.2f}%)")
print(f"R²: {r2_rfr:.4f}")
print(f"R² Ajustado: {r2_ajustada_rfr:.4f}") 

MAE: 32242.8729
MSE: 2500726638.0465
RMSE: 50007.2659
MAPE: 0.1794 (o 17.94%)
R²: 0.8075
R² Ajustado: 0.8071


### MLPRegressor

In [19]:
from sklearn.neural_network import MLPRegressor

mlp = Pipeline(steps=[
    ('preprocessor', preprossesor),
    ('model', MLPRegressor(
        hidden_layer_sizes=(150, 50),
        activation='relu',
        solver='adam',
        learning_rate_init=0.003,
        max_iter=2500,
        random_state=777,
    ))
])

mlp.fit(x_train, y_train)
y_pred_mlp = mlp.predict(x_test)

mae_mlp = mean_absolute_error(y_test, y_pred_mlp)
mse_mlp = mean_squared_error(y_test, y_pred_mlp)
rmse_mlp = root_mean_squared_error(y_test, y_pred_mlp)
r2_mlp = r2_score(y_test, y_pred_mlp)
mape_mlp = mean_absolute_percentage_error(y_test, y_pred_mlp)

n = x_test.shape[0]
p = mlp.named_steps['preprocessor'].transform(x_test).shape[1]
r2_ajustada_mlp = 1 - ((1 - r2_mlp) * (n - 1) / (n - p - 1))

print(f"MAE: {mae_mlp:.4f}")
print(f"MSE: {mse_mlp:.4f}")
print(f"RMSE: {rmse_mlp:.4f}")
print(f"MAPE: {mape_mlp:.4f} (o {mape_mlp * 100:.2f}%)")
print(f"R²: {r2_mlp:.4f}")
print(f"R² Ajustado: {r2_ajustada_mlp:.4f}")

MAE: 37459.0339
MSE: 3042714807.8787
RMSE: 55160.8086
MAPE: 0.2068 (o 20.68%)
R²: 0.7658
R² Ajustado: 0.7651


### Resumen

Luego de comparar cuatro modelos, se determinó que Random Forest Regressor ofrece el mejor desempeño. Para optimizar los resultados, se realizó un proceso de ingeniería de características estructurado mediante un pipeline, lo que garantizó la consistencia en las transformaciones y previno la fuga de datos. El preprocesamiento incluyó la estandarización de ciertas variables, la aplicación de One-Hot Encoding para variables categóricas y el uso de escalamiento MinMax para normalizar las magnitudes numéricas, asegurando que las diferencias de escala no afecten la precisión de las predicciones.
